# T8 features check (features_v1) — Issue #28
Owner: Zahraa. Reviewer: Fatima (fallback Raghad).

Checks only: this notebook reads `features_v1.parquet` (made by `src/features.py`) and `sales_clean_v1.parquet`, and changes nothing.
Every check stops with an error (`assert`) if a rule is broken. Run top to bottom (Runtime > Restart session and run all).

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/sic-project-restockai'
%cd /content
!rm -rf {REPO}
!git clone -q -b zahraa/t08-features https://github.com/raghad612/sic-project-restockai.git {REPO}
%cd {REPO}

In [ ]:
import numpy as np
import pandas as pd
from src.config import SALES_CLEAN, FEATURES

f = pd.read_parquet(FEATURES)          # output of src/features.py
clean = pd.read_parquet(SALES_CLEAN)   # input (T7)
TRAIN_END = pd.Timestamp("2017-04-30")
KEYS = ["date", "store_nbr", "item_nbr"]

print("rows clean:", len(clean), "| rows features:", len(f))
assert len(f) == len(clean), "row count changed"
assert not f.duplicated(KEYS).any(), "duplicate store-item-day rows"
print("columns:", list(f.columns))

## 1. Target
`target = unit_sales` with negatives (returns) set to 0. Every negative becomes a 0, so zeros after = zeros before + negatives, and positive values are unchanged.

In [ ]:
neg = (f["unit_sales"] < 0).sum()
zeros_before = (f["unit_sales"] == 0).sum()
zeros_after = (f["target"] == 0).sum()
print("negatives:", neg, "| zeros before:", zeros_before, "| zeros after:", zeros_after)
assert (f["target"] >= 0).all()
assert zeros_after == zeros_before + neg
pos = f["unit_sales"] > 0
assert (f.loc[pos, "target"] == f.loc[pos, "unit_sales"]).all()
print("target: PASS")

## 2. Calendar
weekday 0 = Monday. Compared with the day name pandas gives for each date.

In [ ]:
names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
d = f.drop_duplicates("date")
assert (d["weekday"].map(dict(enumerate(names))) == d["date"].dt.day_name()).all()
assert (d["month"] == d["date"].dt.month).all() and (d["day_of_month"] == d["date"].dt.day).all()
print(f[["weekday", "day_of_month", "month", "week_of_year"]].agg(["min", "max"]))
print("calendar: PASS")

## 3. Holiday flags (T6 rules)
Celebrated days only (no Event, Work Day or transferred=True row). The holiday table has 34 national dates in the training part (20 day-off + 14 shopping, same total as T6) and 22 local store-days (same as T6).
On the sales rows we expect 17 day-off dates, not 20: 3 of them (2015-12-25, 2016-01-01, 2016-12-25) are chain closures with no sales rows, so the flag has no row to sit on. 2017-01-01 is not counted because it was transferred to 2017-01-02 (its Transfer row).

In [ ]:
tr = f[f["date"] <= TRAIN_END]
nat = tr.groupby("date")[["hol_national_dayoff", "hol_national_shopping"]].max()
n_dayoff, n_shop = int(nat["hol_national_dayoff"].sum()), int(nat["hol_national_shopping"].sum())
n_local = tr.loc[tr["hol_local"] == 1, ["store_nbr", "date"]].drop_duplicates().shape[0]
print("national day-off dates:", n_dayoff, "| shopping dates:", n_shop, "| local store-days:", n_local)
assert (n_dayoff, n_shop, n_local) == (17, 14, 22), "holiday counts differ from T6"

def flags(date, store):
    r = f[(f["date"] == date) & (f["store_nbr"] == store)].iloc[0]
    return (int(r["hol_national_dayoff"]), int(r["hol_national_shopping"]), int(r["hol_local"]))

assert flags("2016-12-24", 44) == (0, 1, 0)   # Additional (shopping day)
assert flags("2016-05-24", 44) == (0, 0, 0)   # transferred=True: original date, not celebrated
assert flags("2016-05-27", 44) == (1, 0, 0)   # its Transfer row: the real day off
assert flags("2017-04-13", 39)[2] == 1        # local holiday in Cuenca, store 39 is in Cuenca
assert flags("2017-04-13", 44)[2] == 0        # same day, store in another city
print("holiday flags: PASS")

## 4. By hand: 4 store-items, 10 rows each
Lags and rolling means are recomputed here from the raw target with a plain filter (slow but obvious), and compared with the saved columns.
- store 44, item 115894: has a missing day (2016-12-25, chain closure).
- the store-item with the most missing days that are not chain closures (if any: T7 zero-filled every other gap, so this is expected to find none, and the closure day above is the missing-day case).
- 2 random store-items.

In [ ]:
closures = pd.to_datetime(["2015-12-25", "2016-01-01", "2016-12-25", "2017-01-01"])

def by_hand(row, k=None, w=None):
    s = f[(f["store_nbr"] == row.store_nbr) & (f["item_nbr"] == row.item_nbr)]
    if k is not None:                                       # lag k: same store-item, date - k
        v = s.loc[s["date"] == row.date - pd.Timedelta(days=k), "target"]
        return v.iloc[0] if len(v) else np.nan
    win = s[(s["date"] >= row.date - pd.Timedelta(days=w)) & (s["date"] < row.date)]
    return win["target"].mean()                              # today excluded: date < row.date

# store-item with the most non-closure gaps
g = f.sort_values(KEYS[1:] + ["date"])
gap = g.groupby(["store_nbr", "item_nbr"])["date"].diff().dt.days.fillna(1)
g = g.assign(gap=gap)
odd = g[(g["gap"] > 1) & ~((g["gap"] == 2) & (g["date"] - pd.Timedelta(days=1)).isin(closures))]
print("store-days with a gap that is not a closure:", len(odd))
pairs = [(44, 115894)]
if len(odd):
    pairs.append(tuple(int(x) for x in odd.groupby(["store_nbr", "item_nbr"]).size().idxmax()))
allp = f[["store_nbr", "item_nbr"]].drop_duplicates()
rest = allp.sample(min(10, len(allp)), random_state=42)
pairs += [p for p in (tuple(int(x) for x in q) for q in rest.itertuples(index=False)) if p not in pairs][:4 - len(pairs)]
print("store-items checked:", pairs)

In [ ]:
cols = ["date", "target", "lag_1", "lag_7", "roll_mean_7", "hol_national_dayoff", "hol_national_shopping", "hol_local"]
for store, item in pairs:
    s = f[(f["store_nbr"] == store) & (f["item_nbr"] == item)]
    gaps = s[s["date"].diff().dt.days > 1]["date"]
    centre = gaps.iloc[0] if len(gaps) else s["date"].iloc[len(s) // 2]
    ten = s[s["date"] >= centre - pd.Timedelta(days=5)].head(10)
    for k in [1, 7, 14]:
        assert np.allclose(ten[f"lag_{k}"], ten.apply(lambda r: by_hand(r, k=k), axis=1), equal_nan=True), (store, item, k)
    for w in [7, 28]:
        assert np.allclose(ten[f"roll_mean_{w}"], ten.apply(lambda r: by_hand(r, w=w), axis=1), equal_nan=True), (store, item, w)
    print(f"store {store}, item {item}: lags and rolling means match by hand")
    display(ten[cols])

## 5. Today is not included (small example)
Store 44, item 115894, 2016-12-26: the day before (Dec 25) has no row. lag_1 must be NaN (not Dec 24's value), and roll_mean_7 must be the mean of Dec 19–24 (6 days), without Dec 26 itself.

In [ ]:
s = f[(f["store_nbr"] == 44) & (f["item_nbr"] == 115894)].set_index("date")
expected = s.loc["2016-12-19":"2016-12-24", "target"].mean()
print("Dec 19-24 targets:", s.loc["2016-12-19":"2016-12-24", "target"].tolist(), "-> mean", round(expected, 4))
assert np.isnan(s.loc["2016-12-26", "lag_1"])
assert np.isclose(s.loc["2016-12-26", "roll_mean_7"], expected)
s.loc["2016-12-17":"2016-12-28", ["target", "lag_1", "lag_7", "roll_mean_7"]]

## 6. 300 random rows: lags and rolling means recomputed by hand

In [ ]:
test = f.sample(300, random_state=42)
for k in [1, 7, 14]:
    assert np.allclose(test[f"lag_{k}"], test.apply(lambda r: by_hand(r, k=k), axis=1), equal_nan=True), k
for w in [7, 28]:
    assert np.allclose(test[f"roll_mean_{w}"], test.apply(lambda r: by_hand(r, w=w), axis=1), equal_nan=True), w
print("300 / 300 rows match for lag_1, lag_7, lag_14, roll_mean_7, roll_mean_28")

## 7. No future information near the split (2017-04-30)
3 store-items x 3 days (Apr 30, May 1, May 2). Every lag and rolling mean is recomputed by hand from earlier dates only (`date < row date`) and must match. Store-items are chosen among those that have rows on all 3 days, so the check is never empty.

In [ ]:
days = pd.to_datetime(["2017-04-30", "2017-05-01", "2017-05-02"])
on_all = f[f["date"].isin(days)].groupby(["store_nbr", "item_nbr"]).size()
on_all = [tuple(int(x) for x in p) for p in on_all[on_all == 3].index]
near_pairs = [(44, 115894)] if (44, 115894) in on_all else []
near_pairs += [p for p in pd.Series(on_all).sample(min(10, len(on_all)), random_state=42) if p not in near_pairs][:3 - len(near_pairs)]

parts = []
for store, item in near_pairs:
    near = f[(f["store_nbr"] == store) & (f["item_nbr"] == item) & f["date"].isin(days)]
    assert len(near) == 3
    for k in [1, 7, 14]:
        assert np.allclose(near[f"lag_{k}"], near.apply(lambda r: by_hand(r, k=k), axis=1), equal_nan=True), (store, item, k)
    for w in [7, 28]:
        assert np.allclose(near[f"roll_mean_{w}"], near.apply(lambda r: by_hand(r, w=w), axis=1), equal_nan=True), (store, item, w)
    parts.append(near)
print("near the split: 3 store-items x 3 days, all lags and rolling means use earlier days only: PASS")
pd.concat(parts)[["store_nbr", "item_nbr", "date", "target", "lag_1", "lag_7", "roll_mean_7", "roll_mean_28"]]

## 8. fill_after_last_train (step 9b)
False (not null) on non-filled rows, no flagged row after 2017-04-30, every flagged row is a filled zero. The count is compared with the 3,828 that T7 reported (printed, not forced: if it differs, both numbers go in the PR).

In [ ]:
fl = f["fill_after_last_train"]
T7_REPORTED = 3828
print("flagged:", int(fl.sum()), "| T7 reported:", T7_REPORTED, "| difference:", int(fl.sum()) - T7_REPORTED)
assert fl.notna().all() and fl.dtype == bool
assert not (fl & (f["date"] > TRAIN_END)).any()
assert (f.loc[fl, "is_filled"]).all() and (f.loc[fl, "target"] == 0).all()
print("fill_after_last_train: PASS")

## 9. onpromotion (step 10)
**Decision:** float32, 1 = promo, 0 = no promo, NaN on the zero-filled rows only.
**Reason:** we do not know if there was a promotion on a day T7 filled, and LightGBM handles NaN in numeric columns (it does not accept pandas' nullable boolean).
Filling those rows with 0 (False) instead would invent "no promotion" on 159,307 days we know nothing about.
**Pitfall:** NaN appears exactly on the zero-filled rows (target = 0), so a model can learn "NaN means 0 sales". In real use the promotion plan is known, so this pattern would not exist. Left for the ML phase (PR and PROGRESS: Open).

In [ ]:
p = f["onpromotion"]
print(p.value_counts(dropna=False))
assert p.dtype == "float32"
assert (p.isna() == f["is_filled"]).all()
print("onpromotion: PASS")

## 10. Missing values per column (for the PR)

In [ ]:
m = f.isna().sum()
m[m > 0]